In [3]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import logging
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from collections import defaultdict
import pandas as pd
from manifold.decoding.functions.utils import check_config_decoding
import numpy as np
import pickle as pkl
from manifold.decoding.functions import nulldistributions
from communication_subspace.ibl_communication.utils import load_widefield_epoch
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import warnings
from matplotlib import pyplot as plt
warnings.filterwarnings("ignore")
from manifold.utils import get_trial_masks
from glob import glob
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp
import statsmodels.stats.multitest as smm
import seaborn as sns
import pickle as pkl

In [ ]:
%load_ext autoreload 
%autoreload 2

In [4]:
with open("../data/processed/significant_stims_choice.pkl",'rb') as f:
    significant_preloads = pkl.load(f)

In [ ]:
from manifold.widefield_decode import prepare_behavior
from manifold.widefield_ppi import aggregate_by_parent, beryl_mapping


session_id = "f7d46a15-9498-40dc-90da-fb977ce844be"
one = ONE(mode="local")
ssl = SessionLoader(one, session_id)
ssl.load_trials(collection="alf")
trials = ssl.trials.copy()

out = np.nan_to_num(trials.contrastLeft) - np.nan_to_num(trials.contrastRight)
trials["signcont"] = out
trials["stim_side"] = np.sign(trials["signcont"])

_, stim_mask = load_trials_and_mask(one, session_id, exclude_nochoice=False, exclude_unbiased=True)
stimulus_non_zero = trials["signcont"] != 0
stim_mask = stim_mask & stimulus_non_zero

_, choice_mask = load_trials_and_mask(one, session_id, exclude_nochoice=True, exclude_unbiased=True)

joint_mask = stim_mask & choice_mask


target_stim, _ = prepare_behavior(
    trials, session_id, joint_mask, epoch="stim",pseudosessions=1
)
target_choice, _ = prepare_behavior(
    trials, session_id, joint_mask, epoch="choice",pseudosessions=1,true_choice_pseudo=False
)

stim_regions = ["MOB","MOs","MOp"]
choice_regions = ["PL"]

stim_regions =[ [x] for x in stim_regions]
choice_regions = [[x] for x in choice_regions]


2026-07-08 23:35:27,700 [INFO] Preparing behavior targets for epoch: 'stim' with 1 pseudosessions.
2026-07-08 23:35:27,809 [INFO] Preparing behavior targets for epoch: 'choice' with 1 pseudosessions.


In [64]:

config = {'hemisphere':("left", "right")}
stim_data, stim_region_names = load_widefield_epoch(one, session_id, trials, config["hemisphere"], epoch="stim", regions=stim_regions)
choice_data, choice_region_names = load_widefield_epoch(one, session_id, trials, config["hemisphere"], epoch="choice", response_time=True, regions=choice_regions)

parent_mapping = beryl_mapping()
# stim_data, stim_region_names = aggregate_by_parent(stim_data, stim_region_names, parent_mapping)
# choice_data, choice_region_names = aggregate_by_parent(choice_data, choice_region_names, parent_mapping)

In [38]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.decomposition import PCA

In [51]:

def cross_temporal_decode_cv_new(
    X_train_epoch, X_test_epoch, Y_train, Y_test, 
    n_components=5, outer_cv_splits=5, inner_cv_splits=3, random_state=42
):
    """
    X_train_epoch: ntrials x N_features (Stimulus Epoch)
    X_test_epoch: ntrials x M_features (Choice Epoch)
    """
    outer_cv = StratifiedKFold(n_splits=outer_cv_splits, shuffle=True, random_state=random_state)
    inner_cv = StratifiedKFold(n_splits=inner_cv_splits, shuffle=True, random_state=random_state)
    oof_predictions = np.zeros(len(Y_test), dtype=float)
    
    param_grid = {
        "classifier__C": np.logspace(-2, 2, 5),
        "classifier__penalty": ["l2"],
    }
    outer_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_train_epoch, Y_train)):
        

        X_train_fold_A = X_train_epoch[train_idx]
        y_train_fold = Y_train[train_idx]

        X_train_fold_B = X_test_epoch[train_idx] 
        X_test_fold_B = X_test_epoch[test_idx]
        y_test_fold = Y_test[test_idx]

        scaler_A = RobustScaler()
        pca_A = PCA(n_components=n_components, random_state=random_state)
        
        X_train_pca = pca_A.fit_transform(scaler_A.fit_transform(X_train_fold_A))

        scaler_B = RobustScaler()
        pca_B = PCA(n_components=n_components, random_state=random_state)
        
        pca_B.fit(scaler_B.fit_transform(X_train_fold_B))
        X_test_pca = pca_B.transform(scaler_B.transform(X_test_fold_B))

        pipeline = Pipeline([
            ("classifier", LogisticRegression(
                solver="lbfgs", max_iter=1000, class_weight="balanced", random_state=random_state
            ))
        ])

        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="balanced_accuracy",
            n_jobs=1,
        )

        grid_search.fit(X_train_pca, y_train_fold)
        best_model = grid_search.best_estimator_


        y_pred_probs = best_model.predict_proba(X_test_pca)[:, 1]
        y_pred = best_model.predict(X_test_pca)

        oof_predictions[test_idx] = y_pred_probs
        fold_score = balanced_accuracy_score(y_test_fold, y_pred)

        outer_scores.append(fold_score)

    return {
        "oof_predictions": oof_predictions,
        "outer_scores": outer_scores,
        "mean_score": np.mean(outer_scores),
        "std_score": np.std(outer_scores),
        "best_model": best_model,
        "grid":grid_search
    }

In [65]:

results = {}
for region_stim_idx in range(len(stim_data)):
    stim_region = stim_data[region_stim_idx][:, joint_mask, :]
    stim_region = stim_region[1,:]-stim_region[0,:]

    for region_choice_idx in range(len(choice_data)): # should be 1
        choice_region = choice_data[region_choice_idx]
        choice_region = choice_region[1,joint_mask]
        resultx = cross_temporal_decode_cv_new(stim_region, choice_region, target_stim, target_choice,n_components=20)
        key = f'{stim_region_names[region_stim_idx]}, {choice_region_names[region_choice_idx]}'
        results[key] = resultx

In [61]:
def plot_true_decoding_performance(results_dict, region_name="Region A -> Region B"):
 
    scores = results_dict['outer_scores']
    mean_score = results_dict['mean_score']
    std_score = results_dict['std_score']
    oof_preds = results_dict['oof_predictions']

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    ax1 = axes[0]
    
 
    sns.stripplot(y=scores, color="darkgray", size=8, jitter=True, ax=ax1, label="Individual Folds")
    
 
    ax1.errorbar(x=0, y=mean_score, yerr=std_score, fmt='o', color="red", 
                 markersize=10, capsize=5, capthick=2, 
                 label=f"Mean ± STD\n({mean_score:.2f} ± {std_score:.2f})")
    
    ax1.axhline(y=0.5, color="black", linestyle="--", linewidth=1.5, label="Chance (0.5)")
    
    ax1.set_xlim(-0.5, 0.5)
    ax1.set_xticks([]) # Remove x-axis ticks
    ax1.set_ylabel("Balanced Accuracy", fontsize=12)
    ax1.set_title(f"CV Fold Performance", fontsize=14)
    ax1.legend(loc='lower right', frameon=False)

    
    ax2 = axes[1]
    
    
    sns.histplot(oof_preds, bins=20, kde=True, color="steelblue", ax=ax2)
    ax2.axvline(x=0.5, color="black", linestyle="--", linewidth=1.5, label="Decision Boundary (0.5)")
    
    ax2.set_xlabel("Predicted Probability (Class 1)", fontsize=12)
    ax2.set_ylabel("Number of Trials", fontsize=12)
    ax2.set_title("OOF Prediction Confidence", fontsize=14)
    ax2.legend(frameon=False)

    plt.suptitle(f"Decoding Performance: {region_name}", fontsize=16, y=1.05)
    sns.despine()
    plt.tight_layout()
    plt.show()